# YOLOv1


**Unified Detection**

the input image is divided into SXS grid and each grid predicts B bounding boxes.

For each grid cell predicts:
- conditional class probability(If there are 4 classes in the dataset, the conditional class probability tensor will have 4 elements per grid cell.)

For each bounding box predicts:
- x, y, w, h
- confidence score(how likely an object exist in the predicted bounding box)

**Network Design**

- 24 conv layers + 2 Fully Connected Layers
- the conv layers alter between 1X1 conv and 3X3 conv
- use linear activation function for the final layer and all other layers use the leaky RELU(a=0.1)

**Training**

Firsrt, pre-train the first 20 conv laeyers + average-pooling layer and a fully connected layer on the ImageNet. After the pre-trainign is done we drop the fully connected average pooling layer and  fully connected layer.


After pre-training is done, we add 4 conv layers and 2 FC layers. And also increase the input resolution of the network from 224X224 to 448X448.

In [7]:
import torch
import torch.nn as nn

In [10]:
class ConvBlock(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
    super(ConvBlock, self).__init__()
    self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
    self.leaky_relu = nn.LeakyReLU(0.1)

  def forward(self, x):
    return self.leaky_relu(self.conv(x))

In [13]:
class ConvBlock(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
    super(ConvBlock, self).__init__()
    self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
    self.leaky_relu = nn.LeakyReLU(0.1)

  def forward(self, x):
    return self.leaky_relu(self.conv(x))


class YOLO(nn.Module):
  def __init__(self):
    super(YOLO, self).__init__()
    self.layers = nn.Sequential(
        ConvBlock(3, 64, 7, 2, 3),
        nn.MaxPool2d(2, 2),
        ConvBlock(64, 192, 3, 1, 1),
        nn.MaxPool2d(2, 2),
        ConvBlock(192, 128, 1, 1, 0),
        ConvBlock(128, 256, 3, 1, 1),
        ConvBlock(256, 256, 1, 1, 0),
        ConvBlock(256, 512, 3, 1, 1),
        nn.MaxPool2d(2, 2),
        *[ConvBlock(512, 256, 1, 1, 0), ConvBlock(256, 512, 3, 1, 1)] * 4,
        ConvBlock(512, 512, 1, 1, 0),
        ConvBlock(512, 1024, 3, 1, 1),
        nn.MaxPool2d(2, 2),
        ConvBlock(1024, 512, 1, 1, 0),
        ConvBlock(512, 1024, 3, 1, 1),
        ConvBlock(1024, 512, 1, 1, 0),
        ConvBlock(512, 1024, 3, 1, 1),
        ConvBlock(1024, 1024, 3, 1, 1),
        ConvBlock(1024, 1024, 3, 2, 1),
    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(7 * 7 * 1024, 4096),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.5),
        nn.Linear(4096, 7 * 7 * 30)
    )

  def forward(self, x):
      x = self.layers(x)
      x = self.fc(x)
      return x.view(-1, 7, 7, 30)

In [16]:
yolo_model = YOLO()
N, C, H, W = 1, 3, 448, 448
sample_input = torch.randn(N, C, H, W)
sample_output = yolo_model(sample_input)
print("Output shape:", sample_output.shape) # (N, 7, 7, 30)

Output shape: torch.Size([1, 7, 7, 30])
